<a href="https://colab.research.google.com/github/pranathi139/GenAI/blob/main/Week2_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y langchain langchain-community langchain-openai chromadb
!pip install langchain==0.1.16 langchain-community==0.0.34 langchain-openai==0.1.3 chromadb==0.4.24 --quiet


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

from google.colab import files
uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)
documents = loader.load()

In [ ]:
!pip uninstall -y langchain
!pip install langchain==0.1.16 langchain-community langchain-openai --quiet
from langchain.text_splitter import RecursiveCharacterTextSplitter
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
except:
    from langchain_community.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)


In [ ]:
!pip install sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=60,   # ↓ VERY IMPORTANT
    do_sample=False
)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

prompt_template = """


Context:
{context}


"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

In [ ]:
while True:
    query = input("Ask question (type 'exit' to stop): ")

    if query.lower() == "exit":
        break

    result = qa_chain.run(query)

    result = result.strip()

    print("\nFinal Answer:", result, "\n")